<a href="https://colab.research.google.com/github/mayrf/pkasolver/blob/main/notebooks/pka_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# pkasolver — pKa Prediction

This notebook demonstrates microstate pKa prediction using **pkasolver**.

Source code and documentation: https://github.com/mayrf/pkasolver

## 1. Install dependencies

Run this cell once. It installs RDKit, PyTorch (CPU), PyG, and pkasolver.
Expect ~3–5 minutes on a fresh Colab runtime.

In [ ]:
# Install RDKit
!pip install -q rdkit

# Install PyTorch (CPU build — sufficient for inference)
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu

# Install PyTorch Geometric
!pip install -q torch_geometric
!pip install -q torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.3.0+cpu.html

# Install pkasolver (latest from GitHub)
!pip install -q git+https://github.com/mayrf/pkasolver.git

print('Installation complete.')

## 2. Validate installation

In [ ]:
import importlib, sys

required = ['rdkit', 'torch', 'torch_geometric', 'pkasolver']
for pkg in required:
    mod = importlib.import_module(pkg)
    ver = getattr(mod, '__version__', 'unknown')
    print(f'{pkg}: {ver}')

## 3. Predict pKa values

Edit the SMILES string below to predict pKa values for any molecule.

In [ ]:
from rdkit import Chem
from pkasolver.query import QueryModel, calculate_microstate_pka_values, draw_pka_reactions
from IPython.display import display

smiles = 'CC(CC1=CC=CC=C1)NC(=O)C(CCCCN)N'  # @param {type:"string"}

mol = Chem.MolFromSmiles(smiles)
if mol is None:
    raise ValueError(f'Invalid SMILES: {smiles}')

print('Predicting pKa values...')
protonation_states = calculate_microstate_pka_values(mol, only_dimorphite=False)
print(f'Found {len(protonation_states)} protonation state(s).')

img = draw_pka_reactions(protonation_states)
display(img)

## 4. Print pKa table

In [ ]:
print(f'{'Protonated':<50} {'Deprotonated':<50} pKa')
print('-' * 110)
for state in protonation_states:
    p = Chem.MolToSmiles(state.protonated_mol)
    d = Chem.MolToSmiles(state.deprotonated_mol)
    print(f'{p:<50} {d:<50} {state.pka:.2f}')